# 03 — Master selector from symbolic analyses

Combines existing v35/rebuild predictions with analysis candidates only if gates pass. Default is conservative: MC and statement corrections are not applied unless precision gate passes on available gold.

In [ ]:

import json, re, os, math, csv, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [Path('/kaggle/working'), Path('/kaggle/input/datasets/yixuanisthebest/v2333333'), Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'), Path('/mnt/data')]

def find_file(names, required=True):
    if isinstance(names, str): names=[names]
    for name in names:
        p=Path(name)
        if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if not d.exists(): continue
        for name in names:
            p=d/name
            if p.exists(): return p
    if required:
        raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p=find_file(names, required=required)
    if p is None: return None, None
    with open(p,'r',encoding='utf-8') as f: return json.load(f), p

def save_json(obj, path):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f: json.dump(obj,f,ensure_ascii=False,indent=2)
    # reload verify
    with open(path,'r',encoding='utf-8') as f: json.load(f)
    print('saved', path)

def norm_text(s): return re.sub(r'\s+',' ',str(s).strip())
def norm_key(s): return re.sub(r'[^a-z0-9]+',' ',str(s).lower()).strip()

def camel_to_words(s):
    s=re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    return norm_key(s)

def label_norm(x):
    if x is None: return None
    s=str(x).strip()
    if s.upper() in ['A','B','C','D']: return s.upper()
    t=s.title()
    return t if t in ['Yes','No','Unknown','Unparseable'] else s

def is_mc_question(q):
    return bool(re.search(r'\n\s*A\.', str(q))) or all(x in str(q) for x in ['A.','B.'])

def flatten_dataset(rows, dataset_name='dataset'):
    flat=[]
    for ri,row in enumerate(rows):
        qs=row.get('questions') if isinstance(row,dict) else None
        ans=row.get('answers') if isinstance(row,dict) else None
        exps=row.get('explanation') if isinstance(row,dict) else None
        if isinstance(qs, list):
            for qi,q in enumerate(qs):
                flat.append({
                    'flat_id': f'{dataset_name}_{ri}_{qi}',
                    'row_i': ri, 'q_i': qi,
                    'question': q,
                    'gold': label_norm(ans[qi]) if isinstance(ans,list) and qi < len(ans) else None,
                    'explanation': exps[qi] if isinstance(exps,list) and qi < len(exps) else None,
                    'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                    'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                    'idx': row.get('idx'),
                    'is_mc': is_mc_question(q),
                    'is_ynu': not is_mc_question(q),
                })
        elif isinstance(row, dict):
            q=row.get('question') or row.get('query') or ''
            flat.append({
                'flat_id': f'{dataset_name}_{ri}_0', 'row_i': ri, 'q_i': 0,
                'question': q, 'gold': label_norm(row.get('answer') or row.get('gold')),
                'explanation': row.get('explanation'),
                'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                'idx': row.get('idx'), 'is_mc': is_mc_question(q), 'is_ynu': not is_mc_question(q)
            })
    return flat

ATOM_RE = re.compile(r'¬?([A-Z][A-Za-z0-9_]*)\(x\)')

def parse_fol_items(fols):
    facts_pos=set(); facts_neg=set(); exists_pos=set(); exists_neg=set(); impl=[]; raw=[]
    for s in fols or []:
        t=str(s).replace('→','->').replace('∀','forall').replace('∃','exists').replace('¬','~')
        raw.append(str(s))
        # exists conjunctions
        if 'exists' in t:
            for m in re.finditer(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t):
                (exists_neg if m.group(1) else exists_pos).add(m.group(2))
        elif 'forall' in t and '->' not in t:
            m=re.search(r'forall\s*x\s*\(?\s*(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t)
            if m:
                (facts_neg if m.group(1) else facts_pos).add(m.group(2))
        if 'forall' in t and '->' in t:
            parts=t.split('->',1)
            left=parts[0]; right=parts[1]
            ml=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', left)
            mr=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', right)
            if ml and mr:
                a_neg,a=ml[-1]; b_neg,b=mr[0]
                impl.append((a, bool(a_neg), b, bool(b_neg), str(s)))
    return {'facts_pos':facts_pos,'facts_neg':facts_neg,'exists_pos':exists_pos,'exists_neg':exists_neg,'impl':impl,'raw':raw}

def closure(parsed):
    pos=set(parsed['facts_pos']); neg=set(parsed['facts_neg']); paths={('pos',p):f'GIVEN ∀x {p}' for p in pos}; paths.update({('neg',p):f'GIVEN ∀x ¬{p}' for p in neg})
    changed=True
    while changed:
        changed=False
        for a,an,b,bn,raw in parsed['impl']:
            # forward
            if not an and a in pos:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            if an and a in neg:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            # contrapositive for single-antecedent implication: A -> B gives ~B -> ~A
            if not an and not bn and b in neg and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
            if not an and bn and b in pos and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
    return pos,neg,paths

def target_from_question(q):
    # approximate by matching CamelCase words against question tokens; fallback none
    return None

def predicate_candidates(fols):
    preds=set()
    for s in fols or []:
        preds.update(re.findall(r'([A-Z][A-Za-z0-9_]*)\(x\)', str(s)))
    return sorted(preds)

def best_predicate_match(text, predicates):
    txt=norm_key(text)
    best=(0,None)
    for p in predicates:
        words=camel_to_words(p).split()
        if not words: continue
        hit=sum(1 for w in words if w in txt)
        score=hit/len(words)
        # exact full phrase bonus
        if camel_to_words(p) in txt: score=max(score,1.0)
        if score>best[0]: best=(score,p)
    return best[1], best[0]

def qtype(q):
    s=str(q).lower()
    if is_mc_question(q): return 'mc'
    if 'at least one' in s or 'there is at least one' in s or re.search(r'\bsome\b', s): return 'existential'
    if re.search(r'\b(all|every)\b', s): return 'universal'
    if 'statement:' in s and 'if ' in s and ' then ' in s: return 'statement_conditional'
    if 'no ' in s: return 'universal_negative'
    return 'other_ynu'

def answer_from_explanation(exp):
    if not exp: return None
    m=re.findall(r'(?:answer is|final answer\s*[:\-]?)\s*(Yes|No|Unknown|[ABCD])\b', str(exp), flags=re.I)
    return label_norm(m[-1]) if m else None


In [ ]:

OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
DATASETS=['generated_v4style_300','benchmark_v2_1000']
PRED_FILES={
 'generated_v4style_300':['v35_fulltest_preds.json','rebuild_generated_v4style_300_preds.json','full_model_eval_v3_2_preds.json'],
 'benchmark_v2_1000':['v35_benchmark_v2_1000_preds.json','rebuild_benchmark_v2_1000_preds.json'],
}
GATE_MIN_PRECISION=0.90
GATE_MIN_CANDIDATES=5
ALLOW_MC_APPLY=False  # change manually only after audit
ALLOW_STATEMENT_APPLY=False


In [ ]:

def metrics(rows, pred_key='pred_selected'):
    labels=['A','B','C','D','Yes','No','Unknown']
    n=len(rows); correct=sum(r.get(pred_key)==r.get('gold') for r in rows)
    per={}
    for lab in labels:
        tp=sum(r.get(pred_key)==lab and r.get('gold')==lab for r in rows)
        fp=sum(r.get(pred_key)==lab and r.get('gold')!=lab for r in rows)
        fn=sum(r.get(pred_key)!=lab and r.get('gold')==lab for r in rows)
        pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f1=2*pr*rc/(pr+rc) if pr+rc else 0
        per[lab]={'precision':pr,'recall':rc,'f1':f1,'support':sum(r.get('gold')==lab for r in rows),'pred_count':sum(r.get(pred_key)==lab for r in rows)}
    return {'n':n,'correct':correct,'acc':correct/n if n else 0,'macro7':sum(per[l]['f1'] for l in labels)/7,'mc_macro':sum(per[l]['f1'] for l in 'ABCD')/4,'ynu_macro':sum(per[l]['f1'] for l in ['Yes','No','Unknown'])/3,'per_label':per,'pred_distribution':dict(Counter(r.get(pred_key) for r in rows))}

def load_rows(dataset):
    rows,p=load_json(PRED_FILES[dataset], required=True)
    out=[]
    for r in rows:
        rr=dict(r)
        rr['pred_base']=rr.get('pred_v35') or rr.get('pred_final') or rr.get('pred_v32') or rr.get('pred_parsefix') or rr.get('pred')
        rr['pred_selected']=rr['pred_base']
        out.append(rr)
    return out,p

def load_analysis(dataset, kind):
    fn=f'02_{dataset}_{kind}_analysis.json'
    rows,p=load_json(fn, required=False)
    return rows or [], p

def gate_candidates(reps):
    c=[r for r in reps if r.get('candidate') and r.get('posthoc') in ['GOOD','BAD']]
    if not c: return {'pass':False,'n':0,'precision':None,'reason':'no candidates'}
    good=sum(r['posthoc']=='GOOD' for r in c); prec=good/len(c)
    return {'pass': len(c)>=GATE_MIN_CANDIDATES and prec>=GATE_MIN_PRECISION, 'n':len(c),'precision':prec,'good':good,'bad':len(c)-good}

combined={}
for ds in DATASETS:
    try:
        rows,pred_path=load_rows(ds)
    except Exception as e:
        print('skip',ds,e); continue
    byid={r.get('flat_id') or r.get('id'):r for r in rows}
    mc_reports,_=load_analysis(ds,'mc_option_proof')
    st_reports,_=load_analysis(ds,'statement_form')
    mc_gate=gate_candidates(mc_reports); st_gate=gate_candidates(st_reports)
    applied=[]
    if ALLOW_MC_APPLY and mc_gate['pass']:
        for r in mc_reports:
            fid=r.get('flat_id'); cand=r.get('candidate')
            if fid in byid and cand:
                old=byid[fid]['pred_selected']; byid[fid]['pred_selected']=cand; applied.append({'kind':'mc','flat_id':fid,'old':old,'new':cand,'posthoc':r.get('posthoc')})
    if ALLOW_STATEMENT_APPLY and st_gate['pass']:
        for r in st_reports:
            fid=r.get('flat_id'); cand=r.get('candidate')
            if fid in byid and cand:
                old=byid[fid]['pred_selected']; byid[fid]['pred_selected']=cand; applied.append({'kind':'statement','flat_id':fid,'old':old,'new':cand,'posthoc':r.get('posthoc')})
    final=list(byid.values())
    base_metrics=metrics([{**r,'pred_selected':r['pred_base']} for r in rows])
    final_metrics=metrics(final)
    summary={'dataset':ds,'pred_input_path':str(pred_path),'base_metrics':base_metrics,'final_metrics':final_metrics,'mc_gate':mc_gate,'statement_gate':st_gate,'allow_mc_apply':ALLOW_MC_APPLY,'allow_statement_apply':ALLOW_STATEMENT_APPLY,'n_applied':len(applied),'applied_preview':applied[:50], 'decision':'APPLIED_ANALYSIS_CANDIDATES' if applied else 'KEEP_BASE_V35_OR_REBUILD'}
    save_json(final, OUT_DIR/f'03_{ds}_master_selected_preds.json')
    save_json(summary, OUT_DIR/f'03_{ds}_master_selector_summary.json')
    combined[ds]=summary
    print('\n',ds, summary['decision'], 'base',base_metrics['acc'],base_metrics['macro7'],'final',final_metrics['acc'],final_metrics['macro7'])

risk={'version':'v36_v37_master_selector','artifact_risk':'LOW_RELOADED_SAVED_PREDS','overfit_risk':'LOW_BY_DEFAULT_ANALYSIS_ONLY_GATED','underfit_risk':'MC_AND_STATEMENT_REMAIN_UNAPPLIED_UNLESS_GATES_PASS','label_collapse':False,'datasets':combined}
save_json(risk, OUT_DIR/'03_master_risk_report.json')
save_json(combined, OUT_DIR/'03_combined_master_summary.json')
